# Lecture 22 · Probes and World Models: the notebook

*The Anatomy of AI.* In the lecture, Othello-GPT learned Othello from move lists alone, and we found a board inside it, read it with probes and changed it. Here you rebuild every step yourself:

1. **Othello by hand**: the rules, and the same 20,000 random games the lecture used (Plates III, IV, VII).
2. **Othello-GPT by hand**: its weights from Hugging Face and its forward pass, written out; it plays legally (Plates V, VI).
3. **The 512 numbers** of the running total (Plate IX).
4. **A probe is one neuron**, trained by hand for square C5: mine or theirs (Plates X, XIV).
5. **Sixty-four squares, three answers**: the board read from inside (Plates XI, XII).
6. **Block by block, and the control** (Plates XIII, XV, XVI).
7. **Flip a disc inside the model** (Plates XVIII–XX).
8. **The scorecard**: 800 edits (Plate XXI).

Every key number is checked against the lecture's own logged number with `check()`. It runs on a free Colab **T4 GPU** in about 10 minutes (*Runtime → Change runtime type → T4 GPU*). No keys needed.

**Words used here.** A *token* is one item the model reads (here, one move). The *running total* (Lecture 19) is the 512 numbers the model carries for each token, updated by every *block*. Blocks are counted from 1; "after block 6" means the running total once six blocks have run. A *probe* is a small model trained to read something out of those numbers.

In [ ]:
# Pinned installs. Free Colab already has torch; a machine without it gets the pinned version.
import importlib.util, subprocess, sys
def pip(*args):                                   # quiet: pip's output is shown only if an install fails
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *args],
                       capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(r.stdout[-3000:] + r.stderr[-3000:])
if importlib.util.find_spec("torch") is None:
    pip("torch==2.5.1")
pip("numpy==2.0.2", "matplotlib==3.10.0", "requests==2.32.3")

In [ ]:
import io, json, math, os, time, warnings
warnings.filterwarnings("ignore", category=FutureWarning)      # library notices, not results
warnings.filterwarnings("ignore", category=UserWarning)
import numpy as np
import matplotlib.pyplot as plt
import torch

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("running on", DEV)
torch.backends.cuda.matmul.allow_tf32 = False                  # plain float32 everywhere, as the lecture ran
T0 = time.time()

# The course's look: black ink on paper, no colour.
PAPER, INK, INK2 = "#F1EBDD", "#16130F", "#7A7062"
plt.rcParams.update({
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK, "text.color": INK,
    "font.family": "serif", "font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
    "hatch.color": INK, "hatch.linewidth": 0.6, "figure.dpi": 110, "legend.frameon": False,
    "lines.color": INK, "patch.edgecolor": INK, "axes.prop_cycle": plt.cycler(color=[INK, INK2]),
})


def minus(text):
    """Print negative numbers with a real minus sign, as the lecture does."""
    return str(text).replace("-", "\u2212")


def check(name, value, expected, tol=0):
    """Compare a result with the lecture's own number. Prints ✓, or stops the notebook with ✗."""
    number = isinstance(expected, (int, float)) and not isinstance(expected, bool)
    if number:
        ok = abs(float(value) - expected) <= tol
        whole = isinstance(expected, int) and float(value) == int(value)
        shown = f"{int(value):,}" if whole else f"{float(value):.6g}"
        said = f"{expected:,}" if isinstance(expected, int) else expected
    else:
        ok = value == expected
        shown, said = repr(value), repr(expected)
    extra = f" ± {tol:g}" if number and tol else ""
    print(f"{'✓' if ok else '✗'} {name}: {minus(shown)}   (lecture: {minus(said)}{extra})")
    if not ok:
        raise AssertionError(f"{name}: got {shown}, the lecture says {expected}")


def fetch(url, path, what):
    """Download a file once (quietly), and say in one line what came down."""
    import requests
    if not os.path.exists(path):
        t = time.time()
        with requests.get(url, stream=True, timeout=300) as r:
            r.raise_for_status()
            with open(path + ".part", "wb") as f:
                for chunk in r.iter_content(1 << 22):
                    f.write(chunk)
        os.rename(path + ".part", path)
        print(f"downloaded {what} ({os.path.getsize(path) / 1e6:.1f} MB) in {time.time() - t:.0f} s")
    return path
T0_START = time.time()

The lecture's own numbers, copied from its logs (`lectures/22-probes-and-world-models/experiments/raw/`). Every `check()` below compares against one of these.

In [ ]:
LECTURE = {
 "games_played": 32768,
 "kept": 20000,
 "game_18000": [
  19,
  34,
  43,
  18,
  29,
  20,
  11,
  37,
  25,
  33,
  41,
  50,
  59,
  3,
  26,
  57,
  4,
  22,
  12,
  17,
  44,
  53,
  42,
  5,
  10,
  9,
  51,
  40,
  32,
  49,
  8,
  60,
  48,
  58,
  15,
  30,
  2,
  1,
  52,
  61,
  23,
  31,
  24,
  45,
  62,
  46,
  21,
  63,
  39,
  13,
  38,
  14,
  0,
  16,
  6,
  56,
  55,
  47,
  54,
  7
 ],
 "openings_k5": 1396,
 "unique_k7": 0.6486,
 "unique_k9": 0.9898,
 "openings_k12": 20000,
 "positions": 118000,
 "top1_legal": 0.9997,
 "topk_exact": 0.9965,
 "mean_legal": 8.6331,
 "legal_move30": [
  0,
  1,
  2,
  8,
  15,
  16,
  24,
  30,
  38,
  46,
  48,
  52,
  56,
  62
 ],
 "p_A1_move30": 0.0725,
 "norm_after_block6": 22.9,
 "norm_after_block5": 17.797,
 "C5_chances": [
  0.02,
  0.9616,
  0.0184
 ],
 "C5_train": 824421,
 "C5_test": 91200,
 "C5_mine_acc": 0.9868,
 "C5_black_acc": 0.5531,
 "flipped": {
  "linear_mt": [
   0.5194,
   0.7462,
   0.8586,
   0.9221,
   0.9546,
   0.9757,
   0.9878,
   0.9922,
   0.9844
  ],
  "linear_bw": [
   0.4711,
   0.5334,
   0.5333,
   0.5344,
   0.5331,
   0.5309,
   0.5295,
   0.5266,
   0.5098
  ],
  "mlp_bw": [
   0.5198,
   0.7568,
   0.8689,
   0.9281,
   0.958,
   0.9778,
   0.9888,
   0.9936,
   0.9429
  ]
 },
 "occupied_linear_mt_6": 0.9899,
 "placed_occupied": 0.5695,
 "parity_6": 1.0,
 "parity_8": 0.8781,
 "random_flipped_6": 0.5455,
 "B2_shadow": -1.36,
 "B2_read_block8_mirror": 24,
 "p_B6_strength4": 0.0587,
 "legal_after_edit": [
  1,
  2,
  8,
  13,
  15,
  16,
  24,
  30,
  38,
  46,
  48,
  52,
  56,
  62
 ],
 "edits": 800,
 "errors_none": 1.18625,
 "perfect_none": 0.11375,
 "errors_5x4": 0.0775,
 "perfect_5x4": 0.93,
 "perfect_5x1": 0.76125,
 "errors_7x8": 0.7325,
 "p_made_illegal_none": 0.0935,
 "p_made_illegal_5x4": 0.0042
}

## Part 1 · Othello by hand (Plates III, IV, VII)

The board is 64 squares, numbered 0 to 63 row by row; we name them A–H by row and 1–8 by column, as the lecture's boards are drawn. A board is 64 numbers: 1 a black disc, −1 a white disc, 0 empty.

**The rule.** You may play on an empty square if, in some direction, it closes a line of your opponent's discs against one of yours. Every such line flips to your colour. To check a square quickly we work out, once, its *rays*: the squares in each of the 8 directions out to the edge.

In [ ]:
CENTER = (27, 28, 35, 36)                          # the four starting squares are never played
SQUARES = [s for s in range(64) if s not in CENTER]   # the 60 playable squares, in order
DIRS = [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]

def nm(sq):
    """Square 0..63 -> its name: row letter A-H, column 1-8."""
    return "ABCDEFGH"[sq // 8] + str(sq % 8 + 1)

RAYS = []                                          # RAYS[sq]: the squares in each direction, nearest first
for sq in range(64):
    r, c = divmod(sq, 8)
    rays = [[(r + dr * k) * 8 + c + dc * k for k in range(1, 8) if 0 <= r + dr * k < 8 and 0 <= c + dc * k < 8]
            for dr, dc in DIRS]
    RAYS.append([ray for ray in rays if ray])

def flips(board, sq, color):
    """The discs that playing `color` on `sq` would flip."""
    out = []
    for ray in RAYS[sq]:
        line = []
        for s in ray:
            if board[s] == -color:
                line.append(s)                     # an opponent's disc: keep walking
            else:
                if board[s] == color and line:     # closed by one of ours: the line flips
                    out += line
                break                              # empty, or ours: this ray is done
    return out

def legal(board, color):
    return [s for s in range(64) if board[s] == 0 and flips(board, s, color)]

def start_board():
    b = [0] * 64
    b[27], b[28], b[35], b[36] = -1, 1, 1, -1      # D4 white, D5 black, E4 black, E5 white
    return b

def play(board, sq, color):
    b = board.copy()
    for s in flips(b, sq, color):
        b[s] = color
    b[sq] = color
    return b

b0 = start_board()
print("black's legal moves at the start:", [nm(s) for s in legal(b0, 1)])
check("black's legal moves at the start", len(legal(b0, 1)), 4)

In [ ]:
def draw_board(ax, board, rings=(), text=None, title="", shade=None):
    """Discs in ink: black filled, white hollow. rings: squares to ring (legal moves). shade: {sq: darkness 0..1}
    draws a reading instead of discs (dark = mine, pale = theirs)."""
    ax.set_xlim(-0.6, 8); ax.set_ylim(8, -0.6); ax.set_aspect("equal"); ax.axis("off")
    for i in range(9):
        ax.plot([0, 8], [i, i], color=INK, lw=0.7); ax.plot([i, i], [0, 8], color=INK, lw=0.7)
    for i in range(8):
        ax.text(-0.3, i + 0.5, "ABCDEFGH"[i], ha="center", va="center", fontsize=8)
        ax.text(i + 0.5, -0.3, str(i + 1), ha="center", va="center", fontsize=8)
    for s in range(64):
        r, c = divmod(s, 8)
        if shade is not None and s in shade:
            ax.add_patch(plt.Circle((c + .5, r + .5), .36, facecolor=str(1 - shade[s] * 0.92), edgecolor=INK, lw=1))
        elif board[s]:
            ax.add_patch(plt.Circle((c + .5, r + .5), .36, facecolor=INK if board[s] == 1 else PAPER, edgecolor=INK, lw=1.1))
        if s in rings:
            ax.add_patch(plt.Circle((c + .5, r + .5), .36, fill=False, edgecolor=INK, lw=0.8, ls=(0, (2, 2))))
        if text and s in text:
            ax.text(c + .5, r + .5, text[s], ha="center", va="center", fontsize=6.5)
    ax.set_title(title, fontsize=10)

fig, axs = plt.subplots(1, 2, figsize=(7, 3.4))
draw_board(axs[0], b0, rings=legal(b0, 1), title="the start: black's 4 legal moves")
b1 = play(b0, 19, 1)
draw_board(axs[1], b1, title=f"black plays {nm(19)}: one white disc flips")
plt.show()

**Random games.** The lecture played random legal games from the start, black first, and kept only those that ran the full 60 moves with no pass, so whose turn it is always follows from the move number. Game number `seed` uses Python's random generator started at `seed`, so the same code gives the same games: you are about to replay the lecture's 20,000 games exactly, in batches of 4,096 seeds as it did.

In [ ]:
import random
from multiprocessing import Pool

def random_game(seed):
    rng = random.Random(seed)
    board, color, moves, boards, passes = start_board(), 1, [], [], 0
    while True:
        L = legal(board, color)
        if not L:                                  # no move: pass, and the other side tries
            color = -color
            L = legal(board, color)
            if not L:
                break                              # neither side can move: game over
            passes += 1
        m = L[rng.randrange(len(L))]
        board = play(board, m, color)
        moves.append(m)
        boards.append(board)
        color = -color
    return moves, boards, passes

t = time.time()
moves, boards, played, seed = [], [], 0, 0
with Pool(os.cpu_count()) as pool:
    while len(moves) < 20000:
        for mv, bd, ps in pool.map(random_game, range(seed, seed + 4096), chunksize=128):
            played += 1
            if len(mv) == 60 and ps == 0 and len(moves) < 20000:
                moves.append(mv); boards.append(bd)
        seed += 4096
moves = np.array(moves, dtype=np.int64)             # [20000, 60] squares played
boards = np.array(boards, dtype=np.int8)            # [20000, 60, 64] the board after each move
print(f"played {played:,} random games in {time.time() - t:.0f} s; kept {len(moves):,} full games with no pass")
check("random games played", played, LECTURE["games_played"])
check("game 18,000 (the running game) is the lecture's, move for move", moves[18000].tolist() == LECTURE["game_18000"], True)

**Every game is new (Plate VII).** How many of the 20,000 games share their first *k* moves with no other game? After a dozen moves, every game is somewhere no other game has been, so legal moves cannot be looked up; they have to be worked out.

In [ ]:
from collections import Counter
uniq = {}
for k in range(1, 16):
    cnt = Counter(map(tuple, moves[:, :k]))
    uniq[k] = (len(cnt), sum(cnt[tuple(m)] == 1 for m in moves[:, :k]) / len(moves))
for k in (1, 5, 7, 9, 12):
    print(f"first {k:2d} moves: {uniq[k][0]:6,} different openings, {uniq[k][1]:6.1%} of games unique")
check("different openings, first 5 moves", uniq[5][0], LECTURE["openings_k5"])
check("games unique after 7 moves", round(uniq[7][1], 4), LECTURE["unique_k7"])
check("games unique after 9 moves", round(uniq[9][1], 4), LECTURE["unique_k9"])
check("different openings, first 12 moves", uniq[12][0], LECTURE["openings_k12"])

fig, ax = plt.subplots(figsize=(6, 2.6))
ks = list(uniq)
ax.plot(ks, [uniq[k][1] for k in ks], marker="o", ms=3)
ax.set_xlabel("first k moves"); ax.set_ylabel("games no other game shares")
ax.set_yticks([0, 0.5, 1]); ax.set_yticklabels(["0%", "50%", "100%"])
plt.show()

**Moves become tokens (Plate IV).** Othello-GPT reads each move as one number: the 60 playable squares are tokens 1 to 60, in order (0 is kept for "pass", unused here). It reads the first 59 moves of a game and guesses each next one; it never sees a board.

In [ ]:
TOKEN = {sq: i + 1 for i, sq in enumerate(SQUARES)}
toks = np.vectorize(TOKEN.get)(moves[:, :59])       # [20000, 59]
print("the running game's first moves:", " ".join(nm(s) for s in moves[18000, :8]), "...")
print("as tokens:                     ", " ".join(str(t) for t in toks[18000, :8]), "...")

## Part 2 · Othello-GPT by hand (Plates V, VI)

The model is the released synthetic Othello-GPT (Li et al. 2023), in Neel Nanda's TransformerLens conversion on Hugging Face: 8 blocks, a running total of 512 numbers, 8 attention heads per block and 2,048 neurons. We read its weight file and write its forward pass ourselves, the same pieces as Lecture 19's GPT-2:
- **the start of the running total**: the token's arrow plus its position's arrow;
- **each block**: a LayerNorm (here without its own scale: it was folded into the next weights), attention that may only look back, then 2,048 neurons with a smooth gate (GELU); each adds its result to the total;
- **the scores**: a last LayerNorm, then one score per token.

In [ ]:
import torch.nn.functional as F
URL = "https://huggingface.co/NeelNanda/Othello-GPT-Transformer-Lens/resolve/main/synthetic_model.pth"
fetch(URL, "othello_gpt.pth", "Othello-GPT's weights")
W = {k: v.float().to(DEV) for k, v in torch.load("othello_gpt.pth", map_location="cpu", weights_only=True).items()}
for k in ("embed.W_E", "pos_embed.W_pos", "blocks.0.attn.W_Q", "blocks.0.attn.W_O", "blocks.0.mlp.W_in", "unembed.W_U"):
    print(f"{k:22s} {tuple(W[k].shape)}")
print(f"{sum(v.numel() for k, v in W.items() if 'W_' in k or 'b_' in k):,} weights")

def ln(x):
    """LayerNorm with no scale of its own: centre the 512 numbers, divide by their spread."""
    x = x - x.mean(-1, keepdim=True)
    return x / (x.pow(2).mean(-1, keepdim=True) + 1e-5).sqrt()

def block(x, l):
    p = lambda n: W[f"blocks.{l}.{n}"]
    T = x.shape[1]
    h = ln(x)
    q, k, v = (torch.einsum("btd,hde->bhte", h, p(f"attn.W_{c}")) + p(f"attn.b_{c}")[None, :, None] for c in "QKV")
    s = (q @ k.transpose(-1, -2)) / 8                                   # 8 = the square root of 64
    s = s.masked_fill(~torch.tril(torch.ones(T, T, dtype=torch.bool, device=x.device)), float("-inf"))
    z = s.softmax(-1) @ v                                               # each head's mix of earlier moves
    x = x + torch.einsum("bhte,hed->btd", z, p("attn.W_O")) + p("attn.b_O")
    return x + F.gelu(ln(x) @ p("mlp.W_in") + p("mlp.b_in")) @ p("mlp.W_out") + p("mlp.b_out")

@torch.no_grad()
def run(tk, upto=None, edits=None, weights=None):
    """tk: [B, T] tokens. Returns the running total after block `upto` (0 = before any block), or with upto=None
    the 61 scores at every position. edits: {layer: fn(x) -> x}, applied to the running total after that block."""
    global W
    keep, W = W, (weights or W)
    try:
        tk = torch.as_tensor(tk, device=DEV)
        x = W["embed.W_E"][tk] + W["pos_embed.W_pos"][: tk.shape[1]]
        for l in range(9):
            if edits and l in edits:
                x = edits[l](x)
            if upto == l:
                return x
            if l < 8:
                x = block(x, l)
        return ln(x) @ W["unembed.W_U"] + W["unembed.b_U"]
    finally:
        W = keep

**It plays legally (Plate VI).** On the 2,000 held-out games (games 18,000–19,999; 118,000 positions), is the model's top guess a legal move? And are its top *k* guesses exactly the *k* legal moves?

In [ ]:
TEST = range(18000, 20000)
top1 = exact = n_legal = 0
for s in range(18000, 20000, 250):
    order = run(toks[s:s + 250])[..., 1:].argsort(-1, descending=True).cpu().numpy()    # squares by score
    for gi in range(order.shape[0]):
        g = s + gi
        for t in range(59):
            L = set(legal(list(boards[g, t]), -1 if t % 2 == 0 else 1))              # after black's move, white plays
            guess = [SQUARES[j] for j in order[gi, t, :len(L)]]
            top1 += guess[0] in L
            exact += set(guess) == L
            n_legal += len(L)
N = 2000 * 59
print(f"top guess legal: {top1 / N:.2%}   top-k guesses = the k legal moves: {exact / N:.2%}   legal moves on average: {n_legal / N:.2f}")
check("positions checked", N, LECTURE["positions"])
check("top guess legal", round(top1 / N, 4), LECTURE["top1_legal"], 0.0002)
check("top-k guesses are exactly the legal moves", round(exact / N, 4), LECTURE["topk_exact"], 0.0005)
check("legal moves per position", round(n_legal / N, 4), LECTURE["mean_legal"])

**The running example: game 18,000 at move 30.** Black is to play. The model has never seen a board, yet it spreads its guesses over exactly the legal moves, about 7% each (its training games were random, so every legal move was equally likely).

In [ ]:
G0, T0 = 18000, 29                                  # move 30 is position 29, counting from 0
board30 = [int(v) for v in boards[G0, T0]]
legal30 = legal(board30, 1)
p30 = run(toks[G0:G0 + 1])[0, T0, 1:].softmax(-1).cpu().numpy()
print(f"{len(legal30)} legal moves for black:", " ".join(nm(s) for s in legal30))
print("the model's chance on each:", " ".join(f"{nm(s)} {p30[SQUARES.index(s)]:.1%}" for s in legal30))
print(f"chance on all other squares together: {1 - sum(p30[SQUARES.index(s)] for s in legal30):.4%}")
check("legal moves at move 30", legal30, LECTURE["legal_move30"])
check("the model's chance on A1", round(float(p30[SQUARES.index(0)]), 4), LECTURE["p_A1_move30"], 0.0002)
fig, ax = plt.subplots(figsize=(3.6, 3.6))
draw_board(ax, board30, rings=legal30, text={s: f"{p30[SQUARES.index(s)]:.0%}" for s in legal30},
           title="move 30, black to play: the model's chances")
plt.show()

## Part 3 · The 512 numbers (Plate IX)

The model holds nothing but its running total: 512 numbers per move. Here it is after block 6, at move 30.

In [ ]:
tk30 = toks[G0:G0 + 1]
r6 = run(tk30, upto=6)[0, T0]
print(f"after block 6, move 30: 512 numbers, length {r6.norm():.1f}; the first eight:", np.round(r6[:8].cpu().numpy(), 2))
check("length of the running total after block 6 at move 30", round(float(r6.norm()), 1), LECTURE["norm_after_block6"], 0.05)
fig, ax = plt.subplots(figsize=(8, 1.4))
ax.vlines(np.arange(512), 0, r6.cpu().numpy(), lw=0.7)
ax.axhline(0, lw=0.5); ax.set_yticks([]); ax.set_xticks([0, 256, 511]); ax.set_xlabel("the 512 numbers, after block 6")
plt.show()

## Part 4 · A probe is one neuron (Plates X, XIV)

Is C5 *mine* (the player about to move owns it) or *theirs*? A probe for that question is Lecture 1's neuron: 512 weights, one bias, and a sigmoid. We train it the way Lecture 2 did: guess, compare, nudge each weight by (guess − truth) × its input.

First the labels. For every game and position, the board in two descriptions: black/white, and mine/theirs ("mine" = whoever moves next). We also mark the discs that have **flipped** since they were placed: their colour now differs from the player who put them there, so remembering who played where cannot get them right.

In [ ]:
def labels(moves, boards):
    B = boards[:, :59].astype(np.int64)                                  # [N, 59, 64]
    bw = np.where(B == 1, 1, np.where(B == -1, 2, 0))                    # 0 empty, 1 black, 2 white
    nxt = np.where(np.arange(59) % 2 == 0, -1, 1)[None, :, None]         # after black's move, white moves
    mt = np.where(B == 0, 0, np.where(B == nxt, 1, 2))                   # 0 empty, 1 mine, 2 theirs
    N = len(moves)
    first = np.full((N, 64), 99)
    for t in range(60):
        first[np.arange(N), moves[:, t]] = np.minimum(first[np.arange(N), moves[:, t]], t)
    placed_col = np.where(first % 2 == 0, 1, -1)                         # black plays the even moves
    placed = np.where(first[:, None, :] <= np.arange(59)[None, :, None], placed_col[:, None, :], 0)
    flipped = (B != 0) & (B != placed)
    return bw, mt, placed, flipped

bw, mt, placed, flipped = labels(moves, boards)
print(f"held-out discs that have flipped since placed: {flipped[18000:].sum() / (bw[18000:] != 0).sum():.0%} of occupied squares")

def residuals(layer, weights=None):
    """The running total after `layer` blocks for all 20,000 games: [20000, 59, 512], kept in half precision."""
    return torch.cat([run(toks[s:s + 500], upto=layer, weights=weights).half() for s in range(0, len(toks), 500)])

In [ ]:
C5 = 20                                             # row C, column 5
R6 = residuals(6)
occ = torch.tensor(bw[:, :, C5] != 0, device=DEV)  # positions where C5 has a disc
X = R6[occ].float()
mu, sd = X.mean(0), float((X - X.mean(0)).std())
X = ((X - mu) / sd)                                 # standardised: each number centred, one common spread
y_mine = torch.tensor(mt[:, :, C5] == 1, device=DEV)[occ].float()
y_black = torch.tensor(bw[:, :, C5] == 1, device=DEV)[occ].float()
game = torch.arange(20000, device=DEV)[:, None].expand(-1, 59)[occ]
tr, te = game < 18000, game >= 18000
print(f"C5 has a disc at {int(tr.sum()):,} training moments and {int(te.sum()):,} held-out ones")
check("training moments for C5", int(tr.sum()), LECTURE["C5_train"])
check("held-out moments for C5", int(te.sum()), LECTURE["C5_test"])

def train_neuron(Xtr, ytr, Xte, steps=2000, bs=4096, lr=0.5, seed=0, log=()):
    """One neuron, sigmoid(w·x + b), trained by hand: Lecture 2's rule on random batches of 4,096, with the step
    size shrinking smoothly to zero (so it settles instead of bouncing). log: steps at which to keep the neuron's
    total on the held-out moments."""
    g = torch.Generator(device=DEV).manual_seed(seed)
    w = torch.randn(512, device=DEV, generator=g) * 512 ** -0.5
    b = torch.zeros((), device=DEV)
    trace = []
    for step in range(steps + 1):
        if step in log:
            trace.append((step, (Xte @ w + b).clone()))
        if step == steps:
            break
        i = torch.randint(0, len(Xtr), (bs,), device=DEV, generator=g)
        x, t = Xtr[i], ytr[i]
        guess = torch.sigmoid(x @ w + b)
        step_size = lr * 0.5 * (1 + math.cos(math.pi * step / steps))
        w -= step_size * x.T @ (guess - t) / bs    # the slope: (guess - truth) x input, averaged
        b -= step_size * (guess - t).mean()
    return w, b, trace

w_mine, b_mine, trace = train_neuron(X[tr], y_mine[tr], X[te], log=(0, 10, 50, 200, 2000))
for step, total in trace:
    print(f"step {step:5d}: held-out moments read right {((total > 0).float() == y_mine[te]).float().mean():.1%}")
acc_mine = float(((X[te] @ w_mine + b_mine > 0).float() == y_mine[te]).float().mean())
check("one neuron, C5 mine or theirs, held out", round(acc_mine, 4), LECTURE["C5_mine_acc"], 0.006)

In [ ]:
pick = torch.tensor(np.random.default_rng(0).choice(int(te.sum()), 360, replace=False), device=DEV)
fig, axs = plt.subplots(1, 2, figsize=(8, 2.4), sharey=True)
for ax, (step, total) in zip(axs, (trace[0], trace[-1])):
    v, m = total[pick].cpu().numpy(), y_mine[te][pick].cpu().numpy()
    jit = np.random.default_rng(1).uniform(-0.3, 0.3, len(v))
    ax.scatter(v[m == 1], jit[m == 1] + 1, s=6, color=INK, label="mine")
    ax.scatter(v[m == 0], jit[m == 0], s=6, facecolors=PAPER, edgecolors=INK, lw=0.6, label="theirs")
    ax.axvline(0, lw=0.6, ls="--"); ax.set_yticks([0, 1]); ax.set_yticklabels(["theirs", "mine"])
    ax.set_title(f"step {step}: the neuron's total w·x + b", fontsize=10)
plt.show()

**Mine and theirs (Plate XIV).** Ask the same neuron a different question: is C5 *black*? One cut through the 512 numbers cannot do it, because the model stores the board as mine/theirs, and which colour is "mine" swaps every move.

In [ ]:
w_black, b_black, _ = train_neuron(X[tr], y_black[tr], X[te])
acc_black = float(((X[te] @ w_black + b_black > 0).float() == y_black[te]).float().mean())
print(f"one neuron, C5 black or white: {acc_black:.1%} held out (a coin toss is 50%)")
check("one neuron, C5 black or white, held out", round(acc_black, 4), LECTURE["C5_black_acc"], 0.03)
del X

## Part 5 · Sixty-four squares, three answers (Plates XI, XII)

Now every square at once, each with three answers (empty, mine, theirs): 64 × 3 = 192 neurons reading the same 512 numbers, with a softmax per square. We train them as the lecture did: AdamW, three passes over the 1.06 million training positions, batches of 4,096. The two-layer version (512 → 64 ReLU neurons → 3, per square) is ready too, for Part 6.

In [ ]:
import math

def train_probe(X, Y, kind="linear", n_cls=3, epochs=3, H=64, lr=1e-3, bs=4096, seed=0):
    """X [N, 512] standardised, Y [N, S] class ids for S squares. Returns predict(X) and the weights.
    The random start and the batches come from the CPU's generator, as in the lecture's run, so the probes match it."""
    torch.manual_seed(seed)
    N, d = X.shape; S = Y.shape[1]
    rnd = lambda *shape, scale: (torch.randn(*shape) * scale).to(DEV).requires_grad_()
    if kind == "linear":
        Wp = rnd(d, S, n_cls, scale=d ** -0.5)
        bp = torch.zeros(S, n_cls, device=DEV, requires_grad=True)
        params = [Wp, bp]
        f = lambda x: torch.einsum("bd,dsc->bsc", x, Wp) + bp
    else:
        W1 = rnd(S, d, H, scale=d ** -0.5)
        b1 = torch.zeros(S, H, device=DEV, requires_grad=True)
        W2 = rnd(S, H, n_cls, scale=H ** -0.5)
        b2 = torch.zeros(S, n_cls, device=DEV, requires_grad=True)
        params = [W1, b1, W2, b2]
        f = lambda x: torch.einsum("bsh,shc->bsc", torch.relu(torch.einsum("bd,sdh->bsh", x, W1) + b1), W2) + b2
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
    steps = epochs * N // bs
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: 0.5 * (1 + math.cos(math.pi * s / steps)))
    g = torch.Generator().manual_seed(seed)
    for _ in range(steps):
        i = torch.randint(0, N, (bs,), generator=g).to(DEV)
        loss = F.cross_entropy(f(X[i].float()).reshape(-1, n_cls), Y[i].reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    predict = lambda Xq: torch.cat([f(Xq[s:s + 8192].float()).argmax(-1) for s in range(0, len(Xq), 8192)])
    return predict, [p.detach() for p in params]

def standardise(R):
    R = R.reshape(-1, 512).float()
    mu = R.mean(0)
    sd = float((R - mu).std())
    return ((R - mu) / sd).half(), mu, sd

n_tr = 18000 * 59
T_ = lambda a: torch.tensor(np.ascontiguousarray(a).reshape(-1, 64), device=DEV)
OCC, FLP = T_(bw != 0)[n_tr:].bool(), T_(flipped)[n_tr:].bool()
Y_MT, Y_BW = T_(mt).long(), T_(bw).long()

def scores(pred, Y):
    ok = pred == Y
    return {"all": float(ok.float().mean()), "occupied": float(ok[OCC].float().mean()), "flipped": float(ok[FLP].float().mean())}

X6, mu6, sd6 = standardise(R6)
del R6
probe6, P6 = train_probe(X6[:n_tr], Y_MT[:n_tr])
sc = scores(probe6(X6[n_tr:]), Y_MT[n_tr:])
print(f"192 neurons after block 6, held out: occupied squares {sc['occupied']:.1%}, flipped discs {sc['flipped']:.1%}")
check("mine/theirs probe after block 6, occupied squares", round(sc["occupied"], 4), LECTURE["occupied_linear_mt_6"], 0.003)
check("mine/theirs probe after block 6, flipped discs", round(sc["flipped"], 4), LECTURE["flipped"]["linear_mt"][6], 0.004)

**The board, read from inside (Plates XI, XII).** At move 30 of the running game: C5's three totals become chances by softmax, and all 64 squares read together give a board.

In [ ]:
x30 = ((run(tk30, upto=6)[0, T0] - mu6) / sd6)
tot = torch.einsum("d,dsc->sc", x30, P6[0]) + P6[1]                      # [64 squares, 3 totals]
ch = tot.softmax(-1).cpu().numpy()
print("C5's three totals (empty, mine, theirs):", np.round(tot[C5].cpu().numpy(), 2), "-> chances", np.round(ch[C5], 3))
check("C5 read as mine, move 30", round(float(ch[C5, 1]), 4), LECTURE["C5_chances"][1], 0.03)
truth = [0 if v == 0 else (1 if v == 1 else 2) for v in board30]      # black moves next, so black = mine
read = ch.argmax(-1)
wrong = [nm(s) for s in range(64) if read[s] != truth[s]]
print(f"squares misread: {len(wrong)} of 64", wrong)
check("squares misread at move 30", len(wrong), 1, 1)
fig, axs = plt.subplots(1, 2, figsize=(7, 3.5))
draw_board(axs[0], board30, title="the real board, move 30")
draw_board(axs[1], board30, title="read from the running total after block 6",
           shade={s: (ch[s, 1] if ch[s, 0] < 0.5 else 0.0) for s in range(64) if ch[s, 0] < 0.5})
plt.show()

## Part 6 · Block by block, and the control (Plates XIII, XV, XVI)

Train the probes after every block (0 = before any block): the one-neuron mine/theirs probe, the one-neuron black/white probe, and the two-layer black/white probe. Score them on flipped discs, the ones memory cannot get. About two minutes on a T4.

In [ ]:
t = time.time()
curves = {"linear_mt": [], "linear_bw": [], "mlp_bw": []}
PROBES = {}                                         # the mine/theirs probe after every block, for the edits below
for layer in range(9):
    Xl, mu_l, sd_l = (X6, mu6, sd6) if layer == 6 else standardise(residuals(layer))
    for kind, Y in (("linear_mt", Y_MT), ("linear_bw", Y_BW), ("mlp_bw", Y_BW)):
        if (layer, kind) == (6, "linear_mt"):
            pred, prm = probe6, P6
        else:
            pred, prm = train_probe(Xl[:n_tr], Y[:n_tr], kind.split("_")[0])
        curves[kind].append(scores(pred(Xl[n_tr:]), Y[n_tr:])["flipped"])
        if kind == "linear_mt":
            PROBES[layer] = {"W": prm[0], "b": prm[1], "mu": mu_l, "sd": sd_l}
    if layer != 6:
        del Xl
    print(f"after block {layer}: flipped discs read right  mine/theirs {curves['linear_mt'][-1]:.1%}   "
          f"black/white {curves['linear_bw'][-1]:.1%}   two-layer black/white {curves['mlp_bw'][-1]:.1%}")
print(f"({time.time() - t:.0f} s)")
for layer in (0, 1, 4, 6, 8):
    check(f"mine/theirs probe after block {layer}, flipped discs", round(curves["linear_mt"][layer], 4), LECTURE["flipped"]["linear_mt"][layer], 0.01)
check("black/white probe after block 6, flipped discs", round(curves["linear_bw"][6], 4), LECTURE["flipped"]["linear_bw"][6], 0.01)
check("two-layer black/white probe after block 6, flipped discs", round(curves["mlp_bw"][6], 4), LECTURE["flipped"]["mlp_bw"][6], 0.006)
check("two-layer black/white probe after block 8, flipped discs", round(curves["mlp_bw"][8], 4), LECTURE["flipped"]["mlp_bw"][8], 0.02)

**The control (Plate XV).** Maybe the probe, not the model, does the work? Give a network with the same shape but *random* weights the same games, and train the same probe on its running total after block 6. If the board were in the probe, it would read just as well.

In [ ]:
g = torch.Generator().manual_seed(0)
RANDOM = {k: (torch.randn(v.shape, generator=g) * 0.8 / 512 ** 0.5).to(DEV) if k.split(".")[-1].startswith("W_")
          else torch.zeros_like(v) for k, v in W.items()}            # same shapes; weights drawn at random, biases 0
Xr, _, _ = standardise(residuals(6, weights=RANDOM))
pred_r, _ = train_probe(Xr[:n_tr], Y_MT[:n_tr])
random_flipped = scores(pred_r(Xr[n_tr:]), Y_MT[n_tr:])["flipped"]
del Xr
print(f"flipped discs read right after block 6: trained model {curves['linear_mt'][6]:.1%}, random weights {random_flipped:.1%}")
check("the random-weights control, flipped discs", round(random_flipped, 4), LECTURE["random_flipped_6"], 0.03)

fig, ax = plt.subplots(figsize=(6.4, 3))
ax.plot(range(9), curves["linear_mt"], marker="o", ms=3, label="one neuron per answer, mine/theirs")
ax.plot(range(9), curves["mlp_bw"], marker="o", ms=3, ls="--", label="two layers, black/white")
ax.plot(range(9), curves["linear_bw"], marker="o", ms=3, color=INK2, label="one neuron per answer, black/white")
ax.scatter([6], [random_flipped], marker="x", color=INK, zorder=3, label="random weights (block 6)")
ax.set_xlabel("after block"); ax.set_ylabel("flipped discs read right"); ax.set_ylim(0.4, 1.01)
ax.legend(fontsize=8, loc="center right")
plt.show()

## Part 7 · Flip a disc inside the model (Plates XVIII–XX)

Reading is not using: a thermometer reads the room without heating it. To show the model *uses* its board, change the board inside it and watch its guesses.

**The edit.** For square B2, the probe after block 5 has an arrow pointing from "theirs" to "mine": d = (w_mine − w_theirs), made length 1 (in the running total's own units). The running total's *shadow* on that arrow, r·d, says how mine B2 looks. A mirror edit flips the shadow's sign; a stronger one pushes past it:

  r′ = r − (1 + strength) · (r·d) · d

At move 30, B2 is white's, so it is *theirs* (black to play). Flipping it should make A1 illegal and B6 legal.

In [ ]:
def flip_dir(pr, sq):
    d = (pr["W"][:, sq, 1] - pr["W"][:, sq, 2]) / pr["sd"]               # in the running total's raw units
    return d / d.norm()

B2, A1, B6 = 9, 0, 13
d5 = flip_dir(PROBES[5], B2)
r5 = run(tk30, upto=5)[0, T0]
print(f"after block 5 at move 30: the running total has length {r5.norm():.2f}; its shadow on B2's arrow is {r5 @ d5:.2f}")
check("length of the running total after block 5, move 30", round(float(r5.norm()), 3), LECTURE["norm_after_block5"], 0.01)
check("B2's shadow on its arrow", round(float(r5 @ d5), 2), LECTURE["B2_shadow"], 0.25)
b_edit = board30.copy(); b_edit[B2] = -b_edit[B2]
check("legal moves once B2 is flipped", legal(b_edit, 1), LECTURE["legal_after_edit"])

def edit_at(layer, sq, strength, pos=T0):
    d = flip_dir(PROBES[layer], sq)
    def fn(x):
        x = x.clone()
        r = x[:, pos]
        x[:, pos] = r - (1 + strength) * (r @ d)[:, None] * d
        return x
    return {layer: fn}

def read_sq(x, layer, sq):
    pr = PROBES[layer]
    return ((torch.einsum("d,dc->c", (x - pr["mu"]) / pr["sd"], pr["W"][:, sq]) + pr["b"][sq]).softmax(-1))[1].item()

rows = {}
for strength in (0, 1, 2, 4, 8):
    ed = edit_at(5, B2, strength) if strength else {}
    reads = [read_sq(run(tk30, upto=later, edits=ed)[0, T0], later, B2)   # B2 as read after blocks 5..8,
             for later in range(5, 9)]                                     # with the edit in place
    p = run(tk30, edits=ed)[0, T0, 1:].softmax(-1).cpu().numpy()
    rows[strength] = (reads, p[SQUARES.index(A1)], p[SQUARES.index(B6)])
    print(f"strength {strength}: B2 read as mine after blocks 5-8 " + " ".join(f"{v:4.0%}" for v in reads)
          + f"   chance on A1 {rows[strength][1]:.1%}, on B6 {rows[strength][2]:.1%}")
check("a mirror edit: B2 read as mine after block 8 (%)", round(rows[1][0][3] * 100), LECTURE["B2_read_block8_mirror"], 15)
check("strength 4: chance on A1 (now illegal) under 0.2%", bool(rows[4][1] < 0.002), True)
check("strength 4: chance on B6 (now legal)", round(float(rows[4][2]), 4), LECTURE["p_B6_strength4"], 0.03)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 2.8))
for strength, ls in ((1, ":"), (2, "-."), (4, "--"), (8, "-")):
    ax.plot(range(5, 9), rows[strength][0], ls=ls, marker="o", ms=3, label=f"strength {strength}")
ax.plot(range(5, 9), rows[0][0], color=INK2, marker="o", ms=3, label="no edit")
ax.set_xticks(range(5, 9)); ax.set_xlabel("read after block"); ax.set_ylabel("B2 read as mine")
ax.set_ylim(-0.02, 1.02); ax.legend(fontsize=8, ncol=2)
plt.show()

## Part 8 · The scorecard (Plate XXI)

One game proves little. The lecture's test: 800 edits on held-out games, at moves 10–49, each flipping a disc whose flip *changes* which moves are legal. Score each with Li et al.'s measure: the edited board has *k* legal moves; how many are missing from the model's top *k* guesses? The same random draw as the lecture picks the same 800 edits.

In [ ]:
rng = random.Random(0)
cases = []                                           # (game, position, square, legal before, legal after)
while len(cases) < 800:
    g, t = 18000 + rng.randrange(2000), rng.randrange(10, 50)
    nxt = -1 if t % 2 == 0 else 1
    b = [int(v) for v in boards[g, t]]
    L = set(legal(b, nxt))
    occ_sq = [s for s in range(64) if b[s] != 0]
    rng.shuffle(occ_sq)
    found = 0
    for s in occ_sq:
        b2 = b.copy(); b2[s] = -b2[s]
        L2 = set(legal(b2, nxt))
        if L2 and L2 != L:
            cases.append((g, t, s, L, L2)); found += 1
        if found == 2:
            break
cases = cases[:800]
check("edits", len(cases), LECTURE["edits"])
SQI = {sq: j for j, sq in enumerate(SQUARES)}

def scorecard(layer=None, strength=0):
    errs, p_illegal = [], []
    for s in range(0, 800, 200):
        part = cases[s:s + 200]
        tk = toks[[c[0] for c in part]]
        pos = torch.tensor([c[1] for c in part], device=DEV)
        ar = torch.arange(len(part), device=DEV)
        edits = None
        if layer is not None:
            D = torch.stack([flip_dir(PROBES[layer], c[2]) for c in part])
            def fn(x, D=D, pos=pos, ar=ar):
                x = x.clone(); r = x[ar, pos]
                x[ar, pos] = r - (1 + strength) * (r * D).sum(-1, keepdim=True) * D
                return x
            edits = {layer: fn}
        P = run(tk, edits=edits)[ar, pos, 1:].softmax(-1).cpu().numpy()
        for i, (g, t, sq, L, L2) in enumerate(part):
            top = {SQUARES[j] for j in np.argsort(-P[i])[:len(L2)]}
            errs.append(len(L2) - len(top & L2))
            p_illegal += [P[i, SQI[q]] for q in L - L2]
    errs = np.array(errs)
    return errs.mean(), (errs == 0).mean(), float(np.mean(p_illegal))

card = {"no edit": scorecard()}
for strength in (1, 2, 4, 8):
    card[f"block 5, strength {strength}"] = scorecard(5, strength)
for k, (e, z, pi) in card.items():
    print(f"{k:22s} missing legal moves {e:.3f}   perfect edits {z:5.1%}   chance on moves made illegal {pi:.4f}")
check("no edit: missing legal moves", round(card["no edit"][0], 4), LECTURE["errors_none"], 0.01)
check("no edit: perfect", round(card["no edit"][1], 4), LECTURE["perfect_none"], 0.01)
check("block 5, strength 4: missing legal moves", round(card["block 5, strength 4"][0], 4), LECTURE["errors_5x4"], 0.05)
check("block 5, strength 4: perfect edits", round(card["block 5, strength 4"][1], 4), LECTURE["perfect_5x4"], 0.025)
check("block 5, a mirror (strength 1): perfect edits", round(card["block 5, strength 1"][1], 4), LECTURE["perfect_5x1"], 0.05)
check("block 5, strength 4: chance on moves made illegal", round(card["block 5, strength 4"][2], 4), LECTURE["p_made_illegal_5x4"], 0.004)

fig, ax = plt.subplots(figsize=(6.4, 2.6))
names = list(card)
ax.barh(range(len(names)), [card[k][1] for k in names], color=PAPER, edgecolor=INK, hatch="////")
ax.set_yticks(range(len(names))); ax.set_yticklabels(names); ax.invert_yaxis()
ax.set_xlim(0, 1); ax.set_xlabel("edits where the model's top k are exactly the new legal moves")
plt.show()
print(f"total run time so far: {(time.time() - T0_START) / 60:.1f} min")

## Your turn

Each exercise is a TODO cell; worked solutions are at the end.

1. **Whose turn is it?** Train a two-answer probe for "black moves next" (the move number's parity) on the running total after block 6 and after block 8. *Hint: `train_probe(X, Y, n_cls=2)` with `Y` of shape `[N, 1]`.*
2. **A probe can lose to no network at all.** Guess each disc's colour as the colour of whoever placed it (`placed`). Score it on held-out occupied squares and compare with the one-neuron black/white probe. *Hint: compare the guess with `bw`.*
3. **Too late to edit.** Run the scorecard with the edit after block 7 at strength 8. *Hint: `scorecard(7, 8)`; you trained the block-7 probe in Part 6.*
4. **Your own edit.** Back in move 30, flip a different disc that changes the legal moves and see whether the model follows. *Hint: loop over occupied squares, compare `legal()` before and after, then use `edit_at(5, sq, 4)`.*

In [ ]:
# 1. TODO: whose turn is it, after block 6 and after block 8?

In [ ]:
# 2. TODO: the "who placed it" guess on held-out occupied squares

In [ ]:
# 3. TODO: the scorecard with the edit after block 7, strength 8

In [ ]:
# 4. TODO: flip another disc at move 30

## Solutions

In [ ]:
# 1. Whose turn is it?
parity = np.broadcast_to((np.arange(59) % 2)[None, :], (20000, 59)).reshape(-1, 1)   # 1: black moves next
Y_PAR = torch.tensor(np.ascontiguousarray(parity), device=DEV).long()
for layer in (6, 8):
    Xl, _, _ = (X6, mu6, sd6) if layer == 6 else standardise(residuals(layer))
    pred, _ = train_probe(Xl[:n_tr], Y_PAR[:n_tr], n_cls=2)
    acc = float((pred(Xl[n_tr:]) == Y_PAR[n_tr:]).float().mean())
    print(f"after block {layer}: whose turn read right {acc:.1%}")
    check(f"whose turn, after block {layer}", round(acc, 4), LECTURE[f"parity_{layer}"], 0.02)
# After block 8 the model no longer needs to keep whose turn it is: only the next move matters.

In [ ]:
# 2. The "who placed it" guess
guess = T_(np.where(placed == 1, 1, np.where(placed == -1, 2, 0)))[n_tr:]
acc_placed = float((guess == Y_BW[n_tr:])[OCC].float().mean())
print(f"who-placed-it guess, occupied squares: {acc_placed:.1%}; it gets every flipped disc wrong, by definition")
check("who-placed-it guess, occupied squares", round(acc_placed, 4), LECTURE["placed_occupied"], 0.0005)
# The one-neuron black/white probe after block 6 reads 53.9% of occupied squares: worse than this guess. Baselines matter.

In [ ]:
# 3. Too late to edit
e7 = scorecard(7, 8)
print(f"edit after block 7, strength 8: missing legal moves {e7[0]:.2f}, perfect {e7[1]:.0%}")
check("block 7, strength 8: missing legal moves", round(e7[0], 2), round(LECTURE["errors_7x8"], 2), 0.15)
# By block 7 the next-move guess is mostly formed; changing the board there changes little.

In [ ]:
# 4. Flip another disc at move 30
for sq in [s for s in range(64) if board30[s] != 0]:
    b2 = board30.copy(); b2[sq] = -b2[sq]
    L2 = legal(b2, 1)
    if L2 and set(L2) != set(legal30) and sq != B2:
        p = run(tk30, edits=edit_at(5, sq, 4))[0, T0, 1:].softmax(-1).cpu().numpy()
        top = {SQUARES[j] for j in np.argsort(-p)[:len(L2)]}
        print(f"flip {nm(sq)}: {len(set(L2) - top)} of {len(L2)} new legal moves missing from the model's top {len(L2)}")
        break